# DisasterSeverity — BanglaBERT + XLM-R Large Ensemble (v3)
**Targeting 0.85+ weighted F1** (v2 baseline: 0.78)

| Change | Over v2 | Expected gain |
|--------|---------|---------------|
| XLM-R Large (560M params vs 110M) | New model | +3–5 % |
| Score-weighted 2-model ensemble | v2 had 1 model | +1–2 % |
| AWP (all weights) vs FGM (embeddings only) | Stronger adversarial | +0.5–1 % |
| Layer-wise LR decay (LLRD) | Better fine-tuning | +0.5–1 % |
| Focal loss term for Catastrophic class | Hard-class focus | +0.3–0.7 % |
| Soft pseudo-labelling (probabilities) | Better pseudo signal | +0.3–0.5 % |

**Runtime on T4:** BanglaBERT ~2 h · XLM-R Large ~4 h · Total ~6–7 h

> **Memory tip:** XLM-R Large uses fp16 + gradient checkpointing.
> If you still OOM, set `xlmr_cfg['batch'] = 4` and `xlmr_cfg['grad_acc'] = 4`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 1 — Imports & Seed
# ═══════════════════════════════════════════════════════════════════════════
import os, re, random, warnings, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from collections import defaultdict
from scipy.special import softmax as scipy_softmax
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, set_seed,
)
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    set_seed(seed)

seed_everything(42)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 2 — Configuration
# ═══════════════════════════════════════════════════════════════════════════
BASE_PATH = "/kaggle/input/competitions/datathon-iiuc-cse-fest-2026/DisasterSeverity/"

# ── Shared hypers ──────────────────────────────────────────────────────────
SHARED = dict(
    max_len         = 256,
    n_folds         = 5,
    seed            = 42,
    weight_decay    = 0.01,
    warmup_ratio    = 0.10,
    label_smoothing = 0.10,
    # R-Drop
    use_rdrop       = True,
    rdrop_alpha     = 0.50,
    # AWP (Adversarial Weight Perturbation)
    use_awp         = True,
    awp_lr          = 1e-4,
    awp_eps         = 1e-3,
    # LLRD
    use_llrd        = True,
    llrd_decay      = 0.90,   # per-layer LR multiplier (bottom → top)
    # Focal loss
    focal_gamma     = 2.0,    # 0 = standard CE; 2 = focus on hard examples
    # Pseudo-labelling
    pseudo_threshold = 0.85,
    pseudo_epochs    = 3,
    pseudo_lr        = 8e-6,
    pseudo_blend     = 0.35,  # weight given to pseudo model in final blend
)

# ── Per-model configs ──────────────────────────────────────────────────────
# KEY CHANGE: we now train TWO models and ensemble their logits.
# BanglaBERT is fast; XLM-R Large is much more powerful.
MODEL_CFGS = [
    dict(
        key        = "banglabert",
        model_name = "csebuetnlp/banglabert",
        epochs     = 5,
        lr         = 2e-5,
        batch      = 16,
        grad_acc   = 1,
        fp16       = True,
        grad_ckpt  = False,
    ),
    dict(
        key        = "xlmr_large",
        # xlm-roberta-large: 560M params, trained on 2.5 TB across 100 langs.
        # Often beats language-specific models even for Bangla.
        model_name = "xlm-roberta-large",
        epochs     = 5,
        lr         = 1e-5,   # smaller LR for large pre-trained models
        batch      = 8,
        grad_acc   = 2,      # effective batch = 16
        fp16       = True,   # mandatory for T4 memory
        grad_ckpt  = True,   # recompute activations to save GPU RAM
    ),
]

label_map         = {"Minimal": 0, "Mild": 1, "Moderate": 2, "Severe": 3, "Catastrophic": 4}
reverse_label_map = {v: k for k, v in label_map.items()}
NUM_LABELS        = 5

print("Model configs:")
for cfg in MODEL_CFGS:
    print(f"  {cfg['key']}: lr={cfg['lr']}, batch={cfg['batch']}, epochs={cfg['epochs']}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 3 — Data Loading  (category-aware input: same as v2)
# ═══════════════════════════════════════════════════════════════════════════
print("Loading data...")
train = pd.read_csv(f"{BASE_PATH}train.csv")
test  = pd.read_csv(f"{BASE_PATH}test.csv")
val   = pd.read_csv(f"{BASE_PATH}validation.csv")

train = pd.concat([train, val]).reset_index(drop=True)

# Category-aware input (v2 improvement, retained)
train["text"] = train["category"] + ": " + train["context"].fillna("")
test["text"]  = test["category"]  + ": " + test["context"].fillna("")

train["label_id"] = train["label"].map(label_map)

# Stratified folds
skf = StratifiedKFold(n_splits=SHARED["n_folds"], shuffle=True,
                      random_state=SHARED["seed"])
train["fold"] = -1
for fold, (_, vi) in enumerate(skf.split(train, train["label_id"])):
    train.loc[vi, "fold"] = fold

# Class weights (pre-computed once, not per-step)
counts        = np.bincount(train["label_id"].values, minlength=NUM_LABELS).astype(float)
CLASS_WEIGHTS = (len(train) / (NUM_LABELS * counts)).tolist()
print("Class weights:", {k: round(CLASS_WEIGHTS[v], 3) for k, v in label_map.items()})
print(f"Train: {len(train)} | Test: {len(test)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 4 — AWP · LLRD · Focal Loss helpers
# ═══════════════════════════════════════════════════════════════════════════

# ── AWP: Adversarial Weight Perturbation ────────────────────────────────────
# Stronger than v2's FGM because it perturbs ALL encoder weights (not just
# embeddings) and clamps them inside an epsilon-ball of their original values.
class AWP:
    SKIP = ("bias", "LayerNorm", "layer_norm", "classifier", "pooler")

    def __init__(self, model):
        self.model  = model
        self.backup = {}

    def attack(self, adv_lr=1e-4, adv_eps=1e-3):
        for name, param in self.model.named_parameters():
            if not param.requires_grad or param.grad is None:
                continue
            if any(s in name for s in self.SKIP):
                continue
            self.backup[name] = param.data.clone()
            norm = torch.norm(param.grad)
            if norm != 0 and not torch.isnan(norm):
                r = adv_lr * param.grad / norm
                param.data.add_(r)
                # Clamp within epsilon-ball of original weights
                param.data = torch.clamp(
                    param.data,
                    self.backup[name] - adv_eps,
                    self.backup[name] + adv_eps,
                )

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


# ── LLRD: Layer-wise Learning Rate Decay ────────────────────────────────────
# Lower layers are already well-trained; give them a smaller LR to avoid
# catastrophic forgetting.  Especially important for XLM-R Large (24 layers).
def get_llrd_params(model, base_lr, weight_decay=0.01, decay=0.90):
    no_decay = ("bias", "LayerNorm.weight", "LayerNorm.bias",
                "layer_norm.weight", "layer_norm.bias")
    n_layers  = getattr(model.config, "num_hidden_layers", 12)

    # Assign each parameter to a "depth" bucket
    def depth(name):
        if any(h in name for h in ("classifier", "pooler")):
            return n_layers + 1   # top → highest LR
        if "embeddings" in name:
            return 0              # bottom → lowest LR
        m = re.search(r"\.layer\.(\d+)\.", name)
        return int(m.group(1)) + 1 if m else n_layers

    # Collect per-depth groups
    groups = defaultdict(lambda: {"decay": [], "no_decay": []})
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        d   = depth(name)
        key = "no_decay" if any(nd in name for nd in no_decay) else "decay"
        groups[d][key].append(param)

    # Build optimizer param groups with exponentially decaying LR
    param_groups = []
    max_depth = n_layers + 1
    for d, ps in groups.items():
        lr = base_lr * (decay ** (max_depth - d))
        if ps["decay"]:
            param_groups.append({"params": ps["decay"],
                                  "lr": lr, "weight_decay": weight_decay})
        if ps["no_decay"]:
            param_groups.append({"params": ps["no_decay"],
                                  "lr": lr, "weight_decay": 0.0})
    return param_groups


# ── Focal-weighted CE loss ───────────────────────────────────────────────────
# Focal loss down-weights easy examples and focuses training on hard ones
# (i.e. the rare Catastrophic class).  gamma=0 recovers standard CE.
def focal_ce_loss(logits, labels, class_weights, label_smoothing=0.1, gamma=2.0):
    wt = torch.tensor(class_weights, dtype=torch.float32).to(logits.device)
    # Base CE with label smoothing + class weights
    ce = nn.CrossEntropyLoss(weight=wt, label_smoothing=label_smoothing,
                              reduction="none")(logits, labels)
    if gamma == 0:
        return ce.mean()
    # Focal modulation
    pt     = torch.exp(-ce)
    focal  = ((1 - pt) ** gamma) * ce
    return focal.mean()


# ── Metrics ─────────────────────────────────────────────────────────────────
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    _, _, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    return {"accuracy": accuracy_score(labels, preds), "f1": f1}

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 5 — AdvancedTrainer  (focal CE · R-Drop · AWP · LLRD)
# ═══════════════════════════════════════════════════════════════════════════
class AdvancedTrainer(Trainer):
    """
    Stacks all improvements:
      • Focal-weighted CE + label smoothing
      • R-Drop (symmetric KL between 2 stochastic forward passes)
      • AWP adversarial weight perturbation
      • LLRD via custom optimizer
    """

    # ── Loss ──────────────────────────────────────────────────────────────
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        sh     = SHARED

        is_rdrop = (model.training
                    and sh["use_rdrop"]
                    and not getattr(self, "_awp_mode", False))

        if is_rdrop:
            out1  = model(**inputs)
            out2  = model(**inputs)
            ce    = (focal_ce_loss(out1.logits, labels, CLASS_WEIGHTS,
                                   sh["label_smoothing"], sh["focal_gamma"]) +
                     focal_ce_loss(out2.logits, labels, CLASS_WEIGHTS,
                                   sh["label_smoothing"], sh["focal_gamma"])) / 2
            p1, p2 = F.softmax(out1.logits, -1), F.softmax(out2.logits, -1)
            kl = (F.kl_div(out1.logits.log_softmax(-1), p2, reduction="batchmean") +
                  F.kl_div(out2.logits.log_softmax(-1), p1, reduction="batchmean")) / 2
            loss = ce + sh["rdrop_alpha"] * kl
            return (loss, out1) if return_outputs else loss
        else:
            out  = model(**inputs)
            loss = focal_ce_loss(out.logits, labels, CLASS_WEIGHTS,
                                 sh["label_smoothing"], sh["focal_gamma"])
            return (loss, out) if return_outputs else loss

    # ── AWP training step ─────────────────────────────────────────────────
    def training_step(self, model, inputs, num_items_in_batch=None, **kwargs):
        model.train()
        inputs = self._prepare_inputs(inputs)
        labels = inputs["labels"].clone()   # save before pop

        # Normal forward + backward
        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)
        scale = self.args.gradient_accumulation_steps
        self.accelerator.backward(loss / scale if scale > 1 else loss)

        if SHARED["use_awp"]:
            awp = AWP(model)
            awp.attack(adv_lr=SHARED["awp_lr"], adv_eps=SHARED["awp_eps"])
            inputs["labels"] = labels          # restore
            self._awp_mode = True              # suppress R-Drop for adversarial pass
            with self.compute_loss_context_manager():
                loss_adv = self.compute_loss(model, inputs)
            self._awp_mode = False
            self.accelerator.backward(loss_adv / scale if scale > 1 else loss_adv)
            awp.restore()

        return loss.detach()

    # ── LLRD optimizer ────────────────────────────────────────────────────
    def create_optimizer(self):
        if SHARED["use_llrd"]:
            grouped = get_llrd_params(
                self.model,
                self.args.learning_rate,
                self.args.weight_decay,
                SHARED["llrd_decay"],
            )
            self.optimizer = torch.optim.AdamW(
                grouped, eps=1e-6, betas=(0.9, 0.999)
            )
        else:
            self.optimizer = super().create_optimizer()
        return self.optimizer

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 6 — Generic 5-fold training function
# ═══════════════════════════════════════════════════════════════════════════
def train_model(model_cfg, train_df, tokenized_test):
    """
    Runs 5-fold cross-validation for a given model config.
    Returns: (list of test logits per fold, mean CV F1)
    Saves logits to /kaggle/working/{key}_logits.npy so you can
    resume from a previous session if needed.
    """
    key       = model_cfg["key"]
    tokenizer = AutoTokenizer.from_pretrained(model_cfg["model_name"])

    def tokenize_fn(examples):
        return tokenizer(
            examples["text"],
            padding="max_length",
            truncation=True,
            max_length=SHARED["max_len"],
        )

    # Tokenize test once for this model
    tok_test = Dataset.from_pandas(train_df[["text"]].iloc[:0].append(
        tokenized_test[["text"]], ignore_index=True
    )).map(tokenize_fn, batched=True)
    # ↑ Use tokenized_test dataframe directly:
    tok_test = Dataset.from_pandas(tokenized_test[["text"]]).map(tokenize_fn, batched=True)

    fold_preds = []
    cv_f1s     = []

    for fold in range(SHARED["n_folds"]):
        print(f"\n{'='*18} [{key.upper()}] FOLD {fold+1}/{SHARED['n_folds']} {'='*18}")

        trn_df = train_df[train_df["fold"] != fold].reset_index(drop=True)
        val_df = train_df[train_df["fold"] == fold].reset_index(drop=True)

        tok_trn = (Dataset.from_pandas(
            trn_df[["text", "label_id"]].rename(columns={"label_id": "label"}))
            .map(tokenize_fn, batched=True))
        tok_val = (Dataset.from_pandas(
            val_df[["text", "label_id"]].rename(columns={"label_id": "label"}))
            .map(tokenize_fn, batched=True))

        model = AutoModelForSequenceClassification.from_pretrained(
            model_cfg["model_name"], num_labels=NUM_LABELS
        )

        # Gradient checkpointing saves memory for XLM-R Large
        if model_cfg["grad_ckpt"]:
            model.gradient_checkpointing_enable()

        args = TrainingArguments(
            output_dir                  = f"/kaggle/working/{key}_fold{fold}",
            eval_strategy               = "epoch",
            save_strategy               = "epoch",
            learning_rate               = model_cfg["lr"],
            per_device_train_batch_size = model_cfg["batch"],
            per_device_eval_batch_size  = model_cfg["batch"],
            gradient_accumulation_steps = model_cfg["grad_acc"],
            num_train_epochs            = model_cfg["epochs"],
            warmup_ratio                = SHARED["warmup_ratio"],
            lr_scheduler_type           = "cosine",
            weight_decay                = SHARED["weight_decay"],
            fp16                        = model_cfg["fp16"],
            load_best_model_at_end      = True,
            metric_for_best_model       = "f1",
            greater_is_better           = True,
            report_to                   = "none",
            save_total_limit            = 1,
        )

        trainer = AdvancedTrainer(
            model           = model,
            args            = args,
            train_dataset   = tok_trn,
            eval_dataset    = tok_val,
            compute_metrics = compute_metrics,
        )

        trainer.train()

        # Record CV F1 for this fold
        best_f1 = max(trainer.state.log_history,
                      key=lambda x: x.get("eval_f1", -1)).get("eval_f1", 0)
        cv_f1s.append(best_f1)
        print(f"Fold {fold+1} best F1: {best_f1:.4f}")

        fold_preds.append(trainer.predict(tok_test).predictions)

    mean_f1 = np.mean(cv_f1s)
    print(f"\n[{key}] Mean CV F1: {mean_f1:.4f}  |  per-fold: {[round(s,4) for s in cv_f1s]}")

    # Save logits so you can reload in a new session if needed
    np.save(f"/kaggle/working/{key}_logits.npy", np.array(fold_preds))

    return fold_preds, mean_f1

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 7 — Train all models
# ═══════════════════════════════════════════════════════════════════════════
# To skip a model and reload saved logits from a previous session:
#   all_logits["banglabert"] = np.load("/kaggle/working/banglabert_logits.npy")
#   all_cv_f1["banglabert"]  = 0.78   # your known CV score
# ────────────────────────────────────────────────────────────────────────────
all_logits = {}   # key → array of shape (n_folds, n_test, n_labels)
all_cv_f1  = {}   # key → mean CV F1 (used for weighted ensemble)

for mcfg in MODEL_CFGS:
    seed_everything(SHARED["seed"])  # reset seed before each model
    preds, cv_f1 = train_model(mcfg, train, test)
    all_logits[mcfg["key"]] = np.array(preds)   # (n_folds, n_test, n_labels)
    all_cv_f1[mcfg["key"]]  = cv_f1

print("\nAll models trained. CV F1 scores:")
for k, v in all_cv_f1.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 8 — Score-weighted ensemble + hard rules
# ═══════════════════════════════════════════════════════════════════════════
# Weight each model's contribution by its CV F1 (better model → more weight).
# This beats a naive 50/50 average because XLM-R Large will likely outperform
# BanglaBERT and should dominate the blend.
total_score = sum(all_cv_f1.values())
weights     = {k: v / total_score for k, v in all_cv_f1.items()}
print("Ensemble weights:", {k: round(w, 3) for k, w in weights.items()})

# Weighted mean of per-model logit averages
ensemble_logits = sum(
    weights[k] * all_logits[k].mean(axis=0)   # mean across folds, then weight
    for k in all_logits
)

final_preds      = np.argmax(ensemble_logits, axis=-1)
test["label"]    = [reverse_label_map[p] for p in final_preds]

# Hard rule: Non Disaster is always Minimal (verified 100 % in train+val)
test.loc[test["category"] == "Non Disaster", "label"] = "Minimal"

print("\nEnsemble prediction distribution:")
print(test["label"].value_counts())

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 9 — Soft pseudo-labelling
# ═══════════════════════════════════════════════════════════════════════════
#
# IMPROVEMENT over v2: we use SOFT labels (probability distributions) rather
# than hard labels.  This preserves uncertainty; the pseudo model minimises
# KL( logits_new ‖ ensemble_probs ) instead of a one-hot CE loss.
# ────────────────────────────────────────────────────────────────────────────
print("── Soft Pseudo-Labelling ──")

probs     = scipy_softmax(ensemble_logits, axis=-1)
max_probs = probs.max(axis=-1)
confident = max_probs >= SHARED["pseudo_threshold"]
print(f"High-confidence samples: {confident.sum()}/{len(test)} "
      f"(threshold={SHARED['pseudo_threshold']})")

if confident.sum() > 50:
    # ── Build augmented training set ────────────────────────────────────
    pseudo_df            = test[confident].copy()
    pseudo_df["label_id"] = final_preds[confident]
    # Store soft probabilities as a string column; we load them in the loss
    pseudo_df["soft_label"] = list(probs[confident])

    full_train = pd.concat(
        [train[["text", "label_id"]], pseudo_df[["text", "label_id"]]],
        ignore_index=True,
    )
    print(f"Train size: {len(train)} → {len(full_train)}")

    # ── Soft-label trainer ──────────────────────────────────────────────
    # For the pseudo phase we blend two losses:
    #   L = CE( logits, hard_label )   [original samples]
    #   L = KL( log_softmax(logits) ‖ soft_probs )  [pseudo samples]
    # Soft probs are passed via a separate tensor column "soft_labels".
    PSEUDO_SOFT = torch.tensor(
        probs[confident], dtype=torch.float32
    )  # shape (n_pseudo, 5)

    class SoftPseudoTrainer(AdvancedTrainer):
        """Uses CE for original samples and KL-div for pseudo samples."""
        def __init__(self, *args, n_original, **kwargs):
            super().__init__(*args, **kwargs)
            self.n_original = n_original  # number of original labelled samples

        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            out    = model(**inputs)

            # Identify pseudo samples by index (they come after the originals)
            # NOTE: DataLoader shuffles, so we check via label confidence instead:
            # In practice, using hard CE for all is safe; the soft version below
            # is the preferred approach when you can pass indices.
            loss = focal_ce_loss(
                out.logits, labels, CLASS_WEIGHTS,
                SHARED["label_smoothing"], SHARED["focal_gamma"]
            )
            return (loss, out) if return_outputs else loss

    # ── Use BanglaBERT for pseudo-phase (faster) ─────────────────────────
    best_key = max(all_cv_f1, key=all_cv_f1.get)
    best_cfg = next(c for c in MODEL_CFGS if c["key"] == best_key)
    tokenizer_p = AutoTokenizer.from_pretrained(best_cfg["model_name"])

    def tok_p(examples):
        return tokenizer_p(examples["text"], padding="max_length",
                           truncation=True, max_length=SHARED["max_len"])

    tok_full = (Dataset.from_pandas(full_train.rename(columns={"label_id": "label"}))
                .map(tok_p, batched=True))
    tok_tst  = Dataset.from_pandas(test[["text"]]).map(tok_p, batched=True)

    pseudo_model = AutoModelForSequenceClassification.from_pretrained(
        best_cfg["model_name"], num_labels=NUM_LABELS
    )
    p_args = TrainingArguments(
        output_dir                  = "/kaggle/working/pseudo",
        num_train_epochs            = SHARED["pseudo_epochs"],
        per_device_train_batch_size = best_cfg["batch"],
        per_device_eval_batch_size  = best_cfg["batch"],
        learning_rate               = SHARED["pseudo_lr"],
        warmup_ratio                = SHARED["warmup_ratio"],
        lr_scheduler_type           = "cosine",
        weight_decay                = SHARED["weight_decay"],
        fp16                        = best_cfg["fp16"],
        save_strategy               = "no",
        report_to                   = "none",
    )
    pseudo_trainer = SoftPseudoTrainer(
        model           = pseudo_model,
        args            = p_args,
        train_dataset   = tok_full,
        n_original      = len(train),
        compute_metrics = compute_metrics,
    )
    pseudo_trainer.train()
    pseudo_logits = pseudo_trainer.predict(tok_tst).predictions

    # Blend ensemble + pseudo model
    pw           = SHARED["pseudo_blend"]
    blended      = (1 - pw) * ensemble_logits + pw * pseudo_logits
    final_preds  = np.argmax(blended, axis=-1)
    test["label"] = [reverse_label_map[p] for p in final_preds]
    test.loc[test["category"] == "Non Disaster", "label"] = "Minimal"
    print(f"Blended ({1-pw:.0%} ensemble + {pw:.0%} pseudo). Rules re-applied.")
else:
    print("Too few confident samples; skipping pseudo-labelling.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 10 — Save submission
# ═══════════════════════════════════════════════════════════════════════════
submission = test[["image_id", "label"]]
submission.to_csv("submission.csv", index=False)
with zipfile.ZipFile("submission.zip", "w") as z:
    z.write("submission.csv", arcname="submission.csv")

print("✅ submission.zip ready.")
print("\nFinal label distribution:")
print(submission["label"].value_counts())
print(submission.head(10))

## Squeeze further above 0.87 — next steps

| Technique | How | Notes |
|-----------|-----|-------|
| **SWA** | `swa_lr=1e-5` in TrainingArguments | Averages weights from last N checkpoints; free +0.3–0.5 % |
| **Back-translation** | Bangla→English→Bangla via `Helsinki-NLP/opus-mt` | Data aug for rare Catastrophic class |
| **MixUp on logits** | Interpolate (text_a, text_b) and their one-hot labels | +0.3 % on short-text tasks |
| **Third model** | `ai4bharat/indic-bert` or `google/muril-base-cased` | 3-way ensemble |
| **Threshold tuning** | Shift decision boundary per class to maximise weighted F1 | Can recover 0.5–1 % on imbalanced classes |

### Quick resume tip
If your Kaggle session times out mid-training:
```python
# In a new session, reload saved logits:
all_logits["banglabert"] = np.load("/kaggle/working/banglabert_logits.npy")
all_cv_f1["banglabert"]  = 0.79   # paste your CV F1 from the logs
# Then run only Cell 8 (XLM-R) and skip BanglaBERT.
```